# Global Sea Level Rise Using NASA Altimetry

## Introduction
This notebook is a classroom module for OCNG 489/689 **Study the Oceans from Space**. Students use satellite altimetry to build and interpret a global mean sea level (GMSL) time series.

By the end of this activity, you should be able to:
1. Compute a global average from gridded sea surface height using proper area weighting.
1. Fit and interpret a linear trend with least-squares regression.
1. Explain how sampling choices (full record vs one-month-per-year) affect the estimated trend through aliasing.


## Abstract
In this module, students estimate global mean sea level from NASA gridded altimetry data and quantify long-term change with linear regression. The notebook combines physical interpretation (global averaging and area weighting) with statistical modeling (least-squares trend estimation).

## Motivation
- Global mean sea level is a key climate indicator derived from satellite observations.
- A correct global mean requires latitude-dependent area weighting; equal weighting of grid cells is not physically correct.
- Linear regression provides a compact estimate of long-term rate of change (mm/year).
- Under-sampling can alias seasonal/interannual variability into trend estimates, so sampling design must be tested.

## Objectives
- Build a GMSL time series from gridded sea level anomaly fields.
- Derive and apply the least-squares linear regression model ($y=b_0+b_1x+\epsilon$) for sea-level trend estimation.
- Compare trends from full temporal sampling and one-month-per-year sampling.
- Connect trend differences to aliasing and unresolved variability.

## Outline
1. Load the altimetry product and inspect one global sea level anomaly map.
1. Compute area-weighted global mean sea level for each time step.
1. Review least-squares linear regression and estimate the trend with `linregress`.
1. Recompute the trend using sparse sampling (one month per year).
1. Use Dask to compute the full-record trend efficiently and compare with sparse-sampling results.
1. Interpret differences using aliasing theory and ocean variability.


## Analysis Workflow

Use this sequence when teaching or working through the lab in class:
1. Open one example file and visualize global SSHA (standard plot + Cartopy map).
1. Use the `area(...)` module to show why cell area decreases toward the poles.
1. Compute area-weighted global mean sea level for each file.
1. Fit a linear trend and interpret slope/intercept/residuals.
1. Repeat trend estimation with one-month-per-year sampling to demonstrate aliasing sensitivity.
1. Run the all-month calculation with Dask, then compare full vs sparse trend estimates.


## Data products

This lab uses three datasets with different roles in class:

1. **Primary analysis dataset: MEaSURES-SSH (gridded altimetry)**
   - short name: `SEA_SURFACE_HEIGHT_ALT_GRIDS_L4_2SATS_5DAY_6THDEG_V_JPL1812`
   - use in class: compute global mean sea level time series and trend
   - [landing page](https://podaac.jpl.nasa.gov/dataset/SEA_SURFACE_HEIGHT_ALT_GRIDS_L4_2SATS_5DAY_6THDEG_V_JPL1812)

1. **Comparison dataset: reconstructed global mean sea level (GMSL)**
   - short name: `JPL_RECON_GMSL`
   - use in class: compare trend behavior and discuss interpretation limits
   - [landing page](https://podaac.jpl.nasa.gov/dataset/JPL_RECON_GMSL)

1. **Extension dataset (optional exercise): ECCO global mean sea level**
   - short name: `ECCO_L4_GMSL_TIME_SERIES_MONTHLY_V4R4`
   - use in class: transfer the same workflow to a model-based product
   - [landing page](https://doi.org/10.5067/ECTSM-MSL44)


In [ ]:
#load python modules

import xarray as xr
import numpy as np
import pylab as plt
import pandas as pd
#Short_name is used to identify a specific dataset in NASA Earthdata. 
short_name='SEA_SURFACE_HEIGHT_ALT_GRIDS_L4_2SATS_5DAY_6THDEG_V_JPL1812'
short_name='SEA_SURFACE_HEIGHT_ALT_GRIDS_L4_2SATS_5DAY_6THDEG_V_JPL2205' #newer version

## Understand AWS Simple Storage Service (S3) file system (stores all NASA Earthdata)

PO.DAAC cloud (POCLOUD) is a part of Earthdata Cloud. The data are hosted in a S3 bucket on AWS US-West-2. "US-West-2" is a term that refers to the AWS center in Oregon. In this case, the so-called 'Direct-S3 access' only works on the machines hosted in the US-West-2. 

**s3fs** is a pythonic file interface to S3 built on top of [botocore](https://github.com/boto/botocore). s3fs allows typical file-system style operations like cp, mv, ls, du, glob, and put/get of local files to/from S3. Details can be find on its website https://s3fs.readthedocs.io/en/latest/.  

It is important that you set up the **.netrc** file correctly in order to enable the following *init_S3FileSystem* module. The .netrc file should be placed in your home folder. A typical .netrc file has the following content:
```bash
machine urs.earthdata.nasa.gov
    login your_earthdata_username
    password your_earthdata_account_password
 ```

Make sure the permission of the .netrc file is set to 400: ```chmod 400 ~/.netrc``` 

If you do not have or do not remember your Earthdata Login information, go [here](https://urs.earthdata.nasa.gov/users/new) to register or [here](https://urs.earthdata.nasa.gov/reset_passwords/new) to reset password. 

###  AWS credentials with EDL

AWS requires security credentials to access AWS S3.  

With your EDL, you can obtain a temporay S3 credential through https://archive.podaac.earthdata.nasa.gov/s3credentials. It is a 'digital key' to access the Earthdata in AWS cloud. Here is an example:

> {"accessKeyId": "xxxx", "secretAccessKey": "xxxx", "sessionToken": "xxxx", "expiration": "2022-07-22 15:56:34+00:00"}

Further reading: https://docs.aws.amazon.com/general/latest/gr/aws-sec-cred-types.html

In [ ]:
def init_S3FileSystem():
    """
    This routine automatically pull your EDL crediential from .netrc file and use it to obtain an AWS S3 credential through a podaac service accessable at https://archive.podaac.earthdata.nasa.gov/s3credentials
    
    Return:
    =======
    
    s3: an AWS S3 filesystem
    """
    import requests,s3fs
    creds = requests.get('https://archive.podaac.earthdata.nasa.gov/s3credentials').json()
    s3 = s3fs.S3FileSystem(anon=False,
                           key=creds['accessKeyId'],
                           secret=creds['secretAccessKey'], 
                           token=creds['sessionToken'])
    return s3

## Use s3fs.glob to get all file names

The S3FileSystem allows typical file-system style operations like `cp, mv, ls, du, glob`. Once the s3fs file system is established, we can use 'glob' to get all file names from a collection. In this case, the collection S3 path is 
```bash
s3://podaac-ops-cumulus-protected/SEA_SURFACE_HEIGHT_ALT_GRIDS_L4_2SATS_5DAY_6THDEG_V_JPL1812/
```

Using the following will get a list netcdf filenames: 
```
fns=s3sys.glob("s3://podaac-ops-cumulus-protected/SEA_SURFACE_HEIGHT_ALT_GRIDS_L4_2SATS_5DAY_6THDEG_V_JPL1812/*.nc")
```

In [ ]:
s3sys=init_S3FileSystem()

s3path="s3://podaac-ops-cumulus-protected/%s/"%short_name
fns=s3sys.glob(s3path+"*.nc")
print(fns[0])
#Set the time stamps associated with the files
time=pd.date_range(start='1992-10-02',periods=len(fns),freq='5D') 

In [ ]:
print('There are %i files.'%len(fns))

Here is an example file.

In [ ]:
d=xr.open_dataset(s3sys.open(fns[0]))
d

### Plot an example

In [ ]:
plt.figure(figsize=(8,4))
plt.contourf(d['Longitude'],d['Latitude'],d['SLA'][0,...],levels=np.arange(-0.5,0.6,0.05))
plt.ylabel('Latitude')
plt.xlabel('Longitude')
plt.title('Sea Level Anomaly %s'%d.time_coverage_start)
plt.colorbar(label='SLA (m)')

In [ ]:
# Global SSHA map using Cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature

fig = plt.figure(figsize=(10, 5))
ax = plt.axes(projection=ccrs.PlateCarree())
levels = np.arange(-0.5, 0.55, 0.05)
pcm = ax.contourf(
    d["Longitude"], d["Latitude"], d["SLA"][0, ...],
    levels=levels, cmap="RdBu_r", extend="both",
    transform=ccrs.PlateCarree()
)
ax.coastlines(linewidth=0.8)
ax.add_feature(cfeature.LAND, facecolor="lightgray", zorder=0)
ax.set_global()
ax.set_title(f"Global Sea Level Anomaly: {d.time_coverage_start}")
cbar = plt.colorbar(pcm, ax=ax, orientation="vertical", aspect=20, pad=0.05,shrink=0.7)
cbar.set_label("SSHA (m)")
plt.show()


## Calculate the global mean SSHA

The global mean SSH is calculated as follows. 

$SSH_{mean} = \sum \eta(\phi,\lambda)*A(\phi)$, 

where $\phi$ is latitude, $\lambda$ is longitude, $A$ is the area of the grid at latitude $\phi$, and $\eta(\phi,\lambda)*A(\phi)$ is the weighted SLA at $(\phi,\lambda)$. 

The following routine `area` pre-calculates the area as a function of latitude for the 1/6-degree resolution grids. 

In [ ]:
def area(lats, dx=1/6.0):
    """
    Calculate the area associated with a dx by dx degree box centered at each
    latitude specified in `lats`.

    Parameter
    ==========
    lats: list or numpy array of size N
          The latitudes of interest in degrees.

    dx: float, default=1/6.0
        Box size in degrees for both longitude and latitude.

    Return
    =======
    out: ndarray of shape (N,)
         Area values in m^2.
    """
    import numpy as np
    from pyproj import Geod

    # Define WGS84 ellipsoid
    geod = Geod(ellps="WGS84")

    def c_area(lat):
        lons = np.array([-dx/2, dx/2, dx/2, -dx/2])
        lats_box = lat + np.array([-dx/2, -dx/2, dx/2, dx/2])
        area_m2, _ = geod.polygon_area_perimeter(lons, lats_box)
        return abs(area_m2)

    out = []
    for lat in lats:
        out.append(c_area(lat))

    return np.array(out)

def global_mean(fn_s3,s3sys,ssh_area):
    """
    Calculate the global mean given an s3 file of SSH, a s3fs.S3FileSystem, 
    and the ssh_area, which is precalculated to save computing time. 
    Parameter:
    ===========
    fn_s3: S3 file name, e.g., s3://podaac-ops-cumulus-protected/SEA_SURFACE_HEIGHT_ALT_GRIDS_L4_2SATS_5DAY_6THDEG_V_JPL1812/ssh_grids_v1812_1992100212.nc
    s3sys: generated by s3fs.S3FileSystem
    ssh_area: the area size associated with MEaSURES-SSH 1/6-degree resolution product. 
    
    Return
    =======
    dout: scalar
          The global mean sea level (default unit from MEaSURES-SSH: meter)
    """
    with xr.open_dataset(s3sys.open(fn_s3))['SLA'] as d:
        dout=((d*ssh_area).sum()/(d/d*ssh_area).sum()).values
    return dout


### Illustrate The Area Module
This plot shows how the surface area of a fixed $1/6^\circ \times 1/6^\circ$ grid cell decreases toward the poles, which is why global averaging must be area-weighted.


In [ ]:
lat_demo = np.linspace(-89.5, 89.5, 360)
area_demo = area(lat_demo)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(lat_demo, area_demo / 1e6, lw=2, label="Grid-cell area")
ax.set_xlabel("Latitude (deg)")
ax.set_ylabel(r"Area (km$^2$)")
ax.set_title("Area of a 1/6° × 1/6° Cell vs Latitude")
ax.grid(alpha=0.3)

# Compare with idealized spherical scaling cos(latitude)
scale = np.cos(np.deg2rad(lat_demo))
scale = scale / scale.max() * (area_demo / 1e6).max()
ax.plot(lat_demo, scale, "--", lw=1.5, label=r"$\propto \cos(\phi)$ reference")
ax.legend()
plt.show()


In [ ]:
d=xr.open_dataset(s3sys.open(fns[0]))
#pre-calculate the area for reuse
ssh_area=area(d.Latitude.data).reshape(1,-1,1)
print(ssh_area.shape)

In [ ]:
print('The global mean sea level from %s is %7.5f meters.'%(fns[0],global_mean(fns[0],s3sys,ssh_area) ) )

## Demonstrate using a single thread

<div class="alert alert-block alert-success">

Benchmark: using a single thread takes about 17 min to calculate all 1922 files. Here the program is sped up by skipping every 360 days (72 steps). The small EC2 can handle the computation because it involves one file per step. 
</div>



In [ ]:
%%time

#Loop 26-year 5-daily SSH fields (1922 files)
#Skip every 72 files to speed up

result=[]
t_local=time[::72]
for fn in fns[::72]:
    result.append(global_mean(fn,s3sys,ssh_area)*1e3 )
result=np.array(result)
print(t_local)

### Linear Regression: Formulation and Derivation

To estimate sea-level trend, we fit a straight line to time series data:
$$y_i = b_0 + b_1 x_i + \epsilon_i,$$
where $x_i$ is time, $y_i$ is global mean sea level, $b_1$ is the trend (slope), and $b_0$ is the intercept.

Least squares chooses $b_0, b_1$ to minimize
$$S(b_0,b_1)=\sum_{i=1}^{N}\left(y_i-(b_0+b_1x_i)\right)^2.$$
Set partial derivatives to zero ($\partial S/\partial b_0=0$, $\partial S/\partial b_1=0$), solve the normal equations, and obtain
$$b_1=\frac{\sum_i (x_i-\bar{x})(y_i-\bar{y})}{\sum_i (x_i-\bar{x})^2}, \qquad b_0=\bar{y}-b_1\bar{x}.$$

Interpretation:
- $b_1$ gives the estimated long-term sea-level change rate (e.g., mm/year after unit conversion).
- Residuals $\epsilon_i$ contain seasonal, interannual, and measurement variability not explained by a linear trend.
- If sampling is sparse (for example, one month per year), aliasing can change the fitted slope even when the underlying system is unchanged.


In [ ]:
from scipy.stats import linregress

plt.figure(figsize=(14,5))
plt.plot(t_local,result-10,'r-o',label='Global Mean SLA')
tyr=(t_local-t_local[0])/np.timedelta64(1,'D')/365 #convert the number of years

msk=np.isnan(result)
tyr=tyr[~msk]
result=result[~msk]

print(tyr.shape,result.shape)
#Calculate the linear trend using linear regression `linregress`
rate=linregress(tyr[1:],result[1:]) 
print('The estimated sea level rise rate between 1993 and 2018: %5.1fmm/year.'%(rate[0]) )
plt.text(t_local[0],10, 'Linear trend: %5.2fmm/year'%(rate[0]),fontsize=16)
linfit=rate[0]*tyr+rate[1]
plt.plot(t_local[1:],linfit[1:],'k--',label='Linear fit')
plt.legend()
plt.xlabel('Time (year)',fontsize=16)
plt.ylabel('Global Mean SLA (mm)',fontsize=16)

plt.grid(True)
plt.show()


<div class="alert alert-block alert-success">
<b>Quiz</b> </br> 
The global sea level trend from altimetry should be 3.0mm/year. Why did we get 2.5mm/year from the above analysis? Can you get 3.0mm/year by modifying the above code?
    
<b>Hint</b>: The above analysis is aliased. 
    
</div>


### Add the GMSL from Frederikse et al. https://podaac.jpl.nasa.gov/dataset/JPL_RECON_GMSL

In [ ]:
import earthaccess

plt.figure(figsize=(14,5))
plt.plot(t_local,result-10,'r-o',label='altimetry')

plt.xlabel('Time (year)',fontsize=16)
plt.ylabel('Global Mean SLA (meter)',fontsize=16)
plt.grid(True)

# Add JPL reconstructed GMSL via Earthaccess
earthaccess.login(persist=True)
gmsl_results = earthaccess.search_data(short_name='JPL_RECON_GMSL')
if len(gmsl_results) == 0:
    raise ValueError('No JPL_RECON_GMSL granules found.')

def _granule_start(gr):
    umm = gr.umm if hasattr(gr, "umm") else gr["umm"]
    return pd.to_datetime(umm["TemporalExtent"]["RangeDateTime"]["BeginningDateTime"], utc=True)

gmsl_granule = sorted(gmsl_results, key=_granule_start)[-1]
d1 = xr.open_dataset(earthaccess.open([gmsl_granule])[0])
print(d1)

gmsl_var = "global_average_sea_level_change"
if gmsl_var not in d1.variables:
    candidates = [v for v in d1.data_vars if "sea_level" in v.lower() or "gmsl" in v.lower()]
    if len(candidates) == 0:
        raise KeyError(f"Could not find GMSL variable in dataset variables: {list(d1.data_vars)}")
    gmsl_var = candidates[0]

d1[gmsl_var].plot(label='in-situ')
plt.legend()

plt.savefig('gmsl.png')



<div class="alert alert-block alert-success">
<b>Quiz</b> </br> 
The global sea level trend from tide-gauge reconstruction (3.5mm/year) is steeper than altimetry-based analysis (3.0mm/year). Why is that?

<b>Hint</b>: Altimetry-based analysis does not consider vertical land motion. 
    
</div>


## Full-Record Trend With Dask (All Months)

The previous example used sparse sampling to speed up execution. Here we compute the global mean for **all files** and use Dask to parallelize the per-file calculations. We then compare the full-record trend with a one-month-per-year subsample to illustrate sampling effects and aliasing.


In [ ]:
from scipy.stats import linregress
from dask import delayed, compute
from dask.diagnostics import ProgressBar

@delayed
def global_mean_mm(fn):
    return global_mean(fn, s3sys, ssh_area) * 1e3

# Parallel computation over all available 5-day files
tasks = [global_mean_mm(fn) for fn in fns]
with ProgressBar():
    result_all = np.array(compute(*tasks, scheduler="threads"), dtype=float)

time_all = pd.DatetimeIndex(time)
valid_all = np.isfinite(result_all)
t_all = time_all[valid_all]
y_all = result_all[valid_all]
x_all = (t_all - t_all[0]).days / 365.25
rate_all = linregress(x_all, y_all)

# One-month-per-year subsampling: January only
is_january = time_all.month == 1
valid_jan = valid_all & is_january
t_jan = time_all[valid_jan]
y_jan = result_all[valid_jan]
x_jan = (t_jan - t_jan[0]).days / 365.25
rate_jan = linregress(x_jan, y_jan)

print(f"All-month trend (Dask): {rate_all.slope:.2f} mm/year")
print(f"January-only trend:    {rate_jan.slope:.2f} mm/year")

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(t_all, y_all, color="0.6", lw=1.0, label="All 5-day samples")
ax.plot(t_jan, y_jan, "ro", ms=3, alpha=0.7, label="January-only samples")
ax.plot(t_all, rate_all.slope * x_all + rate_all.intercept, "k-", lw=2,
        label=f"All-month fit: {rate_all.slope:.2f} mm/yr")
ax.plot(t_jan, rate_jan.slope * x_jan + rate_jan.intercept, "r--", lw=2,
        label=f"January-only fit: {rate_jan.slope:.2f} mm/yr")
ax.set_title("Global Mean Sea Level: Full Sampling vs One-Month-Per-Year")
ax.set_xlabel("Time")
ax.set_ylabel("Global Mean SLA (mm)")
ax.grid(alpha=0.3)
ax.legend()
plt.show()


***
## Summary

- Satellite altimetry provides a practical route to estimate global mean sea level from gridded products.
- Correct global averaging requires spatial weighting, not a simple arithmetic mean over latitude-longitude grids.
- Trend estimates depend on sampling design; sparse sampling can bias low-frequency signals.

### Conclusions
- The full time series provides the most defensible baseline estimate of long-term sea level trend.
- One-month-per-year sampling can produce a different trend because seasonal/interannual variability is aliased into the inferred low-frequency change.
- Interpreting trend differences requires both physical reasoning and sampling theory.

### Takeaways
- Always document temporal sampling choices when reporting sea level trends.
- Use sensitivity tests (full record vs. subsampled record) to quantify sampling uncertainty.
- Connect statistical trend estimates to known geophysical variability and measurement limitations.


***
## Further reading

- [Use Dask to speed up the computation](https://github.com/podaac/the-coding-club/blob/main/notebooks/MEaSUREs-SSH-dask.ipynb)
- [Calculate the global mean sea level from ECCO](https://github.com/podaac/the-coding-club/blob/main/notebooks/Compute_ECCO_global_mean_SSH.ipynb)